# Audited downsampling: matched 24-epoch reruns
Run in order on a GPU runtime. Upload the updated project Python files to CODE_SOURCE first. Original results remain unchanged. Archived original split files are optional: fresh runs regenerate the same seed-42 assignments from the source dataset. This rerun uses previously inspected test patients, not a new untouched test cohort. Do not tune against test scores.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, math, platform
import importlib.metadata as metadata
from datetime import datetime, timezone
CODE_SOURCE = Path('/content/drive/MyDrive/Improved_UNet_s4994393')
DATA_SOURCE = Path('/content/drive/MyDrive/HipMRI_Study_open')
RUN_ROOT = Path('/content/drive/MyDrive/hipmri_runs')
REFERENCE_BASELINE = RUN_ROOT/'baseline_cached'/'split.json'
REFERENCE_IMPROVED = RUN_ROOT/'improved_cached'/'split.json'
CODE = Path('/content/hipmri_code_downsampled')
SOURCE = Path('/content/HipMRI_Study_open')
DATA = Path('/content/HipMRI_downsampled_half')
CACHE = Path('/content/hipmri_cache_downsampled_half')
EVIDENCE = RUN_ROOT/'downsampling_evidence'
CODE.mkdir(exist_ok=True)
EVIDENCE.mkdir(parents=True, exist_ok=True)
for p in CODE_SOURCE.glob('*.py'):
    shutil.copy2(p, CODE/p.name)
assert all((CODE/n).exists() for n in ['prepare_3d.py','create_split.py','train.py','dataset.py','modules.py','predict.py','evaluate_3d.py'])
for name in ('semantic_MRs','semantic_labels_only'):
    shutil.copytree(DATA_SOURCE/name, SOURCE/name, dirs_exist_ok=True)
    expected = {p.name:p.stat().st_size for p in (DATA_SOURCE/name).glob('*.nii.gz')}
    actual = {p.name:p.stat().st_size for p in (SOURCE/name).glob('*.nii.gz')}
    assert actual == expected and len(actual) == 211, 'Dataset copy incomplete or unexpected files'
os.chdir(CODE)
sys.path.insert(0, str(CODE))
import torch, nibabel, numpy, matplotlib
assert torch.cuda.is_available(), 'Select a GPU runtime'
from prepare_3d import sha256
from dataset import discover_pairs, split_three_way

def run_command(arguments):
    with subprocess.Popen([sys.executable,'-u',*map(str,arguments)], cwd=CODE,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
        status = p.wait()
    if status:
        raise RuntimeError(f'Command failed with exit code {status}; stop here')

def snapshot(label):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    folder = EVIDENCE/(label+'_'+stamp)
    folder.mkdir(exist_ok=True)
    info = {'captured_at':stamp,'python':sys.version,'platform':platform.platform(),
            'packages':{n:metadata.version(n) for n in ('torch','numpy','nibabel','matplotlib')},
            'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),
            'gpu':torch.cuda.get_device_name(0),
            'code_sha256':{p.name:sha256(p) for p in CODE.glob('*.py')}}
    (folder/'environment.json').write_text(json.dumps(info,indent=2))
    (folder/'requirements-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    (folder/'nvidia-smi.txt').write_text(subprocess.check_output(['nvidia-smi'],text=True))
    return folder
snapshot('before_preprocessing')


## Verify the frozen assignments, then preprocess
Scale is fixed at 0.5. Do not overwrite a partial preparation or silently skip failed cases. If preparation fails, diagnose the printed case before using a fresh output directory.


In [ ]:
def names(split):
    return {k:sorted((Path(a).name,Path(b).name) for a,b in pairs) for k,pairs in split.items()}
# Existing experiments retain their exact reference bytes for resume compatibility.
# A fresh checkout can regenerate assignments without committed result exports.
if not REFERENCE_BASELINE.exists():
    preserved = EVIDENCE/'reference_split.json'
    REFERENCE_BASELINE = preserved if preserved.exists() else EVIDENCE/'generated_reference_split.json'
    if not REFERENCE_BASELINE.exists():
        run_command(['create_split.py','--data-root',SOURCE,'--output',REFERENCE_BASELINE])
reference = json.loads(REFERENCE_BASELINE.read_text())
if REFERENCE_IMPROVED.exists():
    assert names(reference) == names(json.loads(REFERENCE_IMPROVED.read_text()))
assert names(reference) == names(split_three_way(discover_pairs(SOURCE), .15, .15, 42))
assert {k:len(v) for k,v in reference.items()} == {'train':136,'validation':48,'test':27}
patient_sets = {k:{Path(a).name.split('_')[0] for a,b in pairs} for k,pairs in reference.items()}
assert all(not patient_sets[a]&patient_sets[b] for a in patient_sets for b in patient_sets if a!=b)
if not DATA.exists():
    run_command(['prepare_3d.py','--data-root',SOURCE,'--output-root',DATA,
                 '--scale','0.5','--reference-split',REFERENCE_BASELINE])
assert (DATA/'preprocessing_manifest.json').exists(), 'Preparation incomplete: do not train'


In [ ]:
manifest = json.loads((DATA/'preprocessing_manifest.json').read_text())
assert manifest['complete'] and manifest['scale']==.5 and manifest['cases']==211
assert manifest['reference_split_sha256']==sha256(REFERENCE_BASELINE)
assert names(reference)==names(split_three_way(discover_pairs(DATA), .15,.15,42))
expected_cases = {Path(a).name for pairs in reference.values() for a,b in pairs}
assert {r['case'] for r in manifest['records']}==expected_cases
for r in manifest['records']:
    assert r['alignment_verified'] and r['labels_before']==r['labels_after']
    assert r['prostate_voxels_before']>0 and r['prostate_voxels_after']>0
    for folder, filename, source_key, output_key in [
        ('semantic_MRs', r['case'], 'source_image_sha256','output_image_sha256'),
        ('semantic_labels_only',r['case'].replace('_LFOV','_SEMANTIC'),'source_mask_sha256','output_mask_sha256')]:
        assert sha256(SOURCE/folder/filename)==r[source_key]
        assert sha256(DATA/folder/filename)==r[output_key]
    assert r['output_shape']==[v//2 for v in r['original_shape']]
    assert numpy.allclose(r['output_spacing'],numpy.array(r['original_spacing'])*2)
ratios = [r['prostate_physical_volume_ratio'] for r in manifest['records']]
print('Verified all 211 pairs and unchanged patient assignments.')
print('Prostate physical-volume ratios: min / median / max',min(ratios),numpy.median(ratios),max(ratios))
for p in DATA.glob('*'):
    if p.suffix in ('.json','.csv','.png'):
        shutil.copy2(p,EVIDENCE/p.name)
if REFERENCE_BASELINE.resolve() != (EVIDENCE/'reference_split.json').resolve():
    shutil.copy2(REFERENCE_BASELINE,EVIDENCE/'reference_split.json')
from IPython.display import display, Image
for p in sorted(DATA.glob('*_preprocessing.png')):
    display(Image(filename=str(p)))
AUDIT_VERIFIED = True


Inspect the training-only before/after panels above for loss of boundary detail. Label survival and volume ratios do not prove preserved contour quality. If a discrepancy needs investigation, stop here; do not select a new scale based on test accuracy.
## One-epoch baseline pilot
The pilot is epoch 1 of the baseline's 24-epoch run, not an additional training epoch. No original-grid weights are loaded.


In [ ]:
def check_run(architecture):
    folder = RUN_ROOT/(architecture+'_downsampled')
    if folder.exists() and any(folder.iterdir()):
        assert (folder/'last.pt').exists(), 'Nonempty run without last.pt: investigate before continuing'
        cfg = json.loads((folder/'config.json').read_text())
        expected = {'architecture':architecture,'data_root':str(DATA),'seed':42,
                    'base_channels':16,'patch_size':[64,64,64],'batch_size':1,
                    'patches_per_volume':8,'workers':2,'window_batch_size':4,
                    'learning_rate':.001,'weight_decay':.0001,'val_every':1,
                    'cache_dir':str(CACHE),'no_amp':False}
        assert all(cfg[k]==v for k,v in expected.items()), 'Existing run configuration differs'
        assert names(json.loads((folder/'split.json').read_text()))==names(reference)
    return folder

def train_to(architecture, epochs):
    assert AUDIT_VERIFIED
    folder = check_run(architecture)
    args = ['train.py','--architecture',architecture,'--data-root',DATA,'--output-dir',folder,
            '--device','cuda','--epochs',epochs,'--patch-size',64,64,64,'--batch-size',1,
            '--patches-per-volume',8,'--base-channels',16,'--workers',2,'--seed',42,
            '--val-every',1,'--cache-dir',CACHE,'--window-batch-size',4,'--profile']
    if (folder/'last.pt').exists():
        state = torch.load(folder/'last.pt', map_location='cpu', weights_only=True)
        completed = state['epoch']+1
        if completed>=epochs:
            print(architecture, 'already completed', completed, 'epochs')
            return folder
        args += ['--resume',folder/'last.pt']
    snapshot(architecture+'_before_training')
    run_command(args)
    assert names(json.loads((folder/'split.json').read_text()))==names(reference)
    return folder
baseline = train_to('baseline',1)
assert (baseline/'best.pt').exists()
pilot = list(csv.DictReader((baseline/'metrics.csv').open()))[0]
assert math.isfinite(float(pilot['train_loss'])) and math.isfinite(float(pilot['dice_5']))
print('Pilot metrics:',pilot)
print('Pilot timing:',list(csv.DictReader((baseline/'performance.csv').open()))[0])


## Resume baseline to 24 total epochs

In [ ]:
baseline = train_to('baseline',24)


## Train improved to 24 epochs
The first execution starts from random initialisation; later executions can resume only this new downsampled run.

In [ ]:
improved = train_to('improved',24)


## Evaluate both frozen best-validation checkpoints
Run after both histories reach 24 epochs. Scores refer to the downsampled reference grid. No model selection or tuning on test results.

In [ ]:
for architecture in ('baseline','improved'):
    folder = check_run(architecture)
    history = list(csv.DictReader((folder/'metrics.csv').open()))
    assert [int(r['epoch']) for r in history]==list(range(1,25)), 'Require exactly 24 completed epochs'
    snapshot(architecture+'_before_evaluation')
    run_command(['evaluate_3d.py','--checkpoint',folder/'best.pt','--data-root',DATA,
                 '--output-dir',folder/'test_final','--device','cuda',
                 '--window-batch-size',4,'--failure-cases',5])
    print(architecture,json.loads((folder/'test_final'/'summary.json').read_text()))


## Download reporting evidence
This archive excludes datasets and model checkpoints. Keep both new best.pt files backed up separately on Drive. Existing results remain the original-grid experiments until these completed exports are reviewed.

In [ ]:
import zipfile
from google.colab import files
archive = Path('/content/downsampled_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in (EVIDENCE,RUN_ROOT/'baseline_downsampled',RUN_ROOT/'improved_downsampled'):
        for p in folder.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.csv','.png','.txt'):
                z.write(p,p.relative_to(RUN_ROOT))
files.download(str(archive))
